# Section 1 — Modules and Packages

**PCAP-31-03 official section** — 6 exam items, 12% of the exam.

Source: [Python Institute PCAP-31-03 syllabus](https://pythoninstitute.org/pcap-exam-syllabus). Explanations below are grounded in the [official Python documentation](https://docs.python.org/3/) (linked per topic) — read the explanation, then run and tweak the code.

---


## 1.1 Import and use modules and packages

A *module* is just a `.py` file; a *package* is a folder of modules with an `__init__.py` file. Python has four import styles, and they behave differently:

- `import module` — loads the module and binds the name `module`. You must write `module.thing`.
- `import module as m` — same, but bound to a shorter alias `m`.
- `from module import thing` — pulls a specific name directly into your namespace; use `thing` with no prefix.
- `from module import *` — pulls *everything* public into your namespace. Avoid this in real code (it silently overwrites names and makes it unclear where a name came from) — but you must recognize it for the exam.

`dir(module)` lists every name defined in a module (handy for exploring what's available). `sys.path` is the list of directories Python searches, in order, when you `import` something — if a module "can't be found," it's not on `sys.path`.

**Exam tip:** `from module import *` only imports names that don't start with an underscore, unless the module defines `__all__` to explicitly control what `*` exports.


In [1]:
import math                 # import module -> use math.sqrt(...)
import math as m            # import ... as ... -> use m.sqrt(...)
from math import sqrt       # from ... import ... -> use sqrt(...) directly
from math import sqrt as s  # combine both: from ... import ... as ...

print(math.sqrt(16), m.sqrt(25), sqrt(36), s(49))

import sys
print(type(sys.path), len(sys.path))   # sys.path: list of directories Python searches
print(sys.path[0])                     # usually the current script's directory

print([name for name in dir(math) if not name.startswith('_')][:10])  # first 10 public names


4.0 5.0 6.0 7.0
<class 'list'> 7
/usr/lib/python310.zip
['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'ceil', 'comb', 'copysign']


**Qualifying nested modules.** Some standard-library modules are themselves inside a package, so importing them requires the *full dotted path* down to the actual module — this is what the syllabus calls "advanced qualifying for nested modules." `import xml.etree.ElementTree` only binds the top-level name `xml` in your namespace; you must still write the full path `xml.etree.ElementTree.fromstring(...)` to reach it (writing just `xml.fromstring(...)` fails, because `xml` itself has no such function — only its nested submodule does). `import xml.etree.ElementTree as ET` avoids repeating the whole path every time.

In [2]:
import xml.etree.ElementTree          # binds only the top-level name 'xml' ...
# ... but to use the nested module, you must still qualify all the way down:
tree = xml.etree.ElementTree.fromstring('<root><child/></root>')
print(tree.tag, [c.tag for c in tree])

import xml.etree.ElementTree as ET     # shorter alias for the fully-qualified nested module
root = ET.fromstring('<a><b/><c/></a>')
print(root.tag, [child.tag for child in root])

try:
    xml.fromstring('<x/>')             # fails: 'xml' package itself has no fromstring()
except AttributeError as e:
    print('AttributeError:', e)


root ['child']
a ['b', 'c']
AttributeError: module 'xml' has no attribute 'fromstring'


## 1.2 Perform evaluations using the math module

Key functions: `ceil()` rounds *up* to the nearest integer, `floor()` rounds *down*, `trunc()` chops off the decimal part (rounds toward zero). For **positive** numbers `floor()` and `trunc()` agree — for **negative** numbers they don't, and this is a classic PCAP trap. `factorial(n)` returns n! (only for non-negative integers). `sqrt(x)` returns the square root. `hypot(x, y)` returns the Euclidean distance from the origin — `sqrt(x**2 + y**2)` — and as of Python 3.8 accepts any number of coordinates (n-dimensional distance).

Docs: [math — Mathematical functions](https://docs.python.org/3/library/math.html)


In [3]:
import math

print(math.ceil(4.1), math.ceil(4.9))     # 5, 5  (always rounds UP)
print(math.floor(4.1), math.floor(4.9))   # 4, 4  (always rounds DOWN)
print(math.trunc(4.9))                     # 4     (chops the decimal part)

# The classic exam trap: floor() vs trunc() on NEGATIVE numbers
print(math.floor(-4.5))   # -5  (down means more negative)
print(math.trunc(-4.5))   # -4  (toward zero, not down)

print(math.factorial(5))          # 120
print(math.sqrt(2))               # 1.4142135623730951
print(math.hypot(3, 4))           # 5.0  (3-4-5 right triangle)
print(math.hypot(1, 2, 2))        # n-dimensional hypot (Python 3.8+)


5 5
4 4
4
-5
-4
120
1.4142135623730951
5.0
3.0


## 1.3 Generate random values using the random module

`random.random()` returns a float in `[0.0, 1.0)`. `random.seed(n)` resets the generator to a reproducible state — same seed, same sequence of "random" numbers every time (useful for testing). `random.choice(seq)` picks one random element from a non-empty sequence. `random.sample(population, k)` picks `k` *unique* elements without replacement (as opposed to `random.choices()`, a different function, which allows repeats).

Docs: [random — Generate pseudo-random numbers](https://docs.python.org/3/library/random.html)


In [4]:
import random

random.seed(42)              # fix the seed so results are reproducible
print(random.random())       # always the same float after seed(42)
print(random.random())       # the NEXT float in that same fixed sequence

random.seed(42)              # reset to the same seed...
print(random.random())       # ...and we get the first float again

print(random.choice(['rock', 'paper', 'scissors']))   # one random pick
print(random.sample(range(1, 50), 6))                  # 6 unique numbers, like a lottery draw


0.6394267984578837
0.025010755222666936
0.6394267984578837
rock
[48, 18, 16, 15, 9, 7]


## 1.4 Discover host platform properties using the platform module

`platform` reports facts about the machine and Python build running the code: `platform.system()` (`'Linux'`, `'Darwin'`, `'Windows'`), `platform.machine()` (CPU architecture, e.g. `'x86_64'`), `platform.processor()`, `platform.python_implementation()` (`'CPython'`, `'PyPy'`, ...), `platform.python_version_tuple()` (version as a tuple of strings), and `platform.version()` (OS build/version string).

Docs: [platform — Access to underlying platform's identifying data](https://docs.python.org/3/library/platform.html)


In [5]:
import platform

print('system:', platform.system())
print('machine:', platform.machine())
print('processor:', platform.processor())
print('python_implementation:', platform.python_implementation())
print('python_version_tuple:', platform.python_version_tuple())
print('platform():', platform.platform())


system: Linux
machine: aarch64
processor: aarch64
python_implementation: CPython
python_version_tuple: ('3', '10', '12')
platform(): Linux-6.8.0-124-generic-aarch64-with-glibc2.35


## 1.5 Create and use user-defined modules and packages

When Python imports a module, it compiles it to bytecode and caches it in a `__pycache__` folder so the next import is faster. Every module has a built-in `__name__` variable: it's `'__main__'` when the file is *run directly*, and the module's own name when it's *imported* by something else — this is why `if __name__ == '__main__':` is the standard way to write code that only runs when a file is executed directly, not when it's imported. A leading underscore (`_thing`) is the *convention* (not enforced) for "private, internal use only." A package is a folder containing an `__init__.py` file (it can be empty) plus one or more module files; `__init__.py` is what makes Python treat the folder as an importable package rather than just a directory.

Below we actually create a tiny module file on disk, import it, and inspect all of this for real.


In [6]:
from pathlib import Path
import sys, importlib

# 1. Write a real module file to disk
module_code = '''
_internal_counter = 0        # leading underscore = "private by convention"

def greet(name):
    return f"Hello, {name}!"

if __name__ == "__main__":
    print("Running greet_mod.py directly:", greet("World"))
else:
    print(f"greet_mod imported; __name__ is {__name__!r}")
'''
Path('greet_mod.py').write_text(module_code)

# 2. Import it like any other module
sys.path.insert(0, '.')
import greet_mod
print(greet_mod.greet('Javier'))
print('module __name__ seen from outside:', greet_mod.__name__)   # NOT '__main__' when imported
print('__pycache__ created:', Path('__pycache__').exists())

# 3. Clean up the demo files
import shutil
Path('greet_mod.py').unlink()
if Path('__pycache__').exists():
    shutil.rmtree('__pycache__')


greet_mod imported; __name__ is 'greet_mod'
Hello, Javier!
module __name__ seen from outside: greet_mod
__pycache__ created: True


**Building a real package (not just a loose module).** A *package* is a directory containing an `__init__.py` file plus one or more modules — `__init__.py` is what tells Python "treat this folder as an importable package," even when it's empty. Packages can nest: a package can contain a subpackage (another folder with its own `__init__.py`), and you reach code inside it with a fully-qualified dotted path, e.g. `import mypkg.utils.helpers`. When you `import` anything, Python **searches** `sys.path` in order (current directory first, then installed-package locations, etc.) looking for a matching top-level package or module — this is exactly why adding a folder to `sys.path` (as we did earlier with `sys.path.insert(0, '.')`) makes Python able to find packages that live there.

Below we build a real two-level package tree on disk (`mypkg/` containing `mypkg/utils/`) and import from the nested subpackage, to make the "directory tree" concept concrete rather than abstract.

In [7]:
from pathlib import Path
import sys, shutil

# Build this tree on disk:
#   demo_mypkg/
#     __init__.py
#     utils/
#       __init__.py
#       helpers.py

base = Path('demo_mypkg')
utils = base / 'utils'
utils.mkdir(parents=True, exist_ok=True)

(base / '__init__.py').write_text("# makes 'demo_mypkg' an importable package\n")
(utils / '__init__.py').write_text("# makes 'demo_mypkg.utils' an importable SUBpackage\n")
(utils / 'helpers.py').write_text(
    "def shout(msg):\n"
    "    return msg.upper() + '!'\n"
)

sys.path.insert(0, '.')                     # add current dir so Python's search finds demo_mypkg
import demo_mypkg.utils.helpers as helpers  # fully-qualified nested package path

print(helpers.shout('found the nested package'))
print('demo_mypkg is a package:', (base / '__init__.py').exists())
print('demo_mypkg.utils is a subpackage:', (utils / '__init__.py').exists())

# Show sys.path actually being searched, in order, to resolve the import
print('current dir is on sys.path:', '.' in sys.path or str(Path('.').resolve()) in sys.path)

# Clean up the demo tree
shutil.rmtree(base)
if Path('__pycache__').exists():
    shutil.rmtree('__pycache__')


FOUND THE NESTED PACKAGE!
demo_mypkg is a package: True
demo_mypkg.utils is a subpackage: True
current dir is on sys.path: True
